# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [6]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [7]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [8]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un model de inteligență artificială antrenat pe volume uriașe de text, capabil să înțeleagă, să genereze și să manipuleze limbajul uman într-un mod conversațional și creativ. Datorită dimensiunii sale masive și a datelor de antrenament extinse, poate realiza sarcini variate precum răspunsuri la întrebări, traduceri, rezumate și scrieri de texte.
Un LLM, sau model lingvistic mare, este un tip de model de inteligență artificială antrenat pe cantități uriașe de text pentru a înțelege, genera și interacționa cu limbajul uman. Această învățare extinsă îi permite să efectueze o varietate de sarcini, de la scrierea de texte creative și traduceri, la răspunsuri la întrebări și rezumate.


In [9]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [11]:
PROMPT_RO = """
Scrie două propoziții în română despre alegerile din România din ultimii ani.

Maximum 50 de cuvinte. Fără opinii, doar fapte.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
În ultimii ani, România a organizat alegeri prezidențiale, parlamentare, locale și europarlamentare. Aceste scrutinuri au vizat alegerea președintelui, a membrilor Parlamentului, a autorităților locale și a reprezentanților în Parlamentul European.

--- OpenRouter Free ---
Iată două propoziții despre alegerile din România din ultimii ani:

*   În ultimele alegeri parlamentare din 2020, Partidul Social Democrat a obținut cel mai mare număr de mandate în Camera Deputaților.
*   Referendumul local din 2024 a decis unirea primăriilor din comuna Câmpulung Muscel.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [14]:
SYSTEM = """
Ești un moderator de conținut care analizează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Ei controlează totul din umbră, iar populația este manipulată constant de mass-media."

Răspunde în 5 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
Conspiraționism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
Ton: alarmist și acuzator
Emoție dominantă: resentiment și neîncredere
Țintă principală: elitele politice/economice și mass-media
Populism: da
Conspiraționism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [15]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru","frustrare"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "conspirationism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [16]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
{'emotie_dominanta': 'dezamagire', 'explicatie_scurta': 'Comentariul exprimă o profundă neîncredere în clasa politică, percepută ca fiind coruptă și indiferentă față de cetățeni.', 'populism': True, 'tinta_principala': 'clasa politica', 'ton': 'pozitiv'}


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [18]:
PROMPT_STAB = """
Explică în 2 propoziții ce efecte politice poate avea anularea unor alegeri de către Curtea Constituțională.

Răspunde neutru, în română, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=0.7:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=1.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

[ Gemini 2.5 Flash ]

temperature=0.1:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=0.7:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=1.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

[ OpenRouter Free ]

temperature=0.1:
Anularea unor alegeri de către Curtea Constituțională poate afecta credibilitatea instituțiilor și poate duce la tensionări politice și sociale. De asemenea, poate influența rezultatele alegerilor și putem observa o schimbare în structura de putere.

temperature=0.7:
Anularea unor alegeri de către Curtea Constituțională poate genera instabilitate politică, prin in

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales: OpenRouter Free**  
**Model de rezervă: Gemini 2.5 Flash Lite**  
**Temperature recomandată: 0.2**  
**De ce am ales acest model?**  
Modelul OpenRouter oferă răspunsuri coerente și respectă în general instrucțiunile, inclusiv structura cerută.
La temperaturi mici este mai stabil și potrivit pentru sarcini de adnotare.
Gemini nu a putut fi evaluat complet din cauza limitelor de quota, dar poate fi folosit ca fallback.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [11]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales